### Import dependencies

In [1]:
from helpers.sporc import (
    SPORCDataset,
    get_all_categories,
    get_main_categories,
    get_main_category,
    get_subcategories_list,
    is_main_category,
    is_subcategory,
    is_valid_category,
)
from pprint import pprint
import matplotlib.pyplot as plt
from typing import Optional
import pandas as pd
import re

In [2]:
sporc = SPORCDataset(
    local_data_dir="data",
    load_samples_only=True,
    load_turns_eagerly=True,
    show_progress=False,
)

INFO:helpers.sporc.dataset:Loading SPORC dataset from local directory: data
INFO:helpers.sporc.dataset:Loading SPORC sample dataset only.
INFO:helpers.sporc.dataset:✓ All required files found
INFO:helpers.sporc.dataset:Loading all records from local files into memory...
INFO:helpers.sporc.dataset:Loading episode_data_sample...
INFO:helpers.sporc.dataset:Loading speaker_turn_data_sample...
INFO:helpers.sporc.dataset:✓ Loaded 210,000 total records from 2 files
INFO:helpers.sporc.dataset:✓ Local dataset loaded successfully in 5.91 seconds
INFO:helpers.sporc.dataset:✓ Dataset loaded successfully with 210000 total records
INFO:helpers.sporc.dataset:Processing dataset into Podcast and Episode objects...
INFO:helpers.sporc.dataset:Separating episode data from speaker turn data...
INFO:helpers.sporc.dataset:✓ Separation completed in 0.12 seconds
INFO:helpers.sporc.dataset:  Episode records: 10,000, Speaker turn records: 200,000
INFO:helpers.sporc.dataset:Grouping episodes by podcast...
INFO:he

In [3]:
# Get all categories
all_categories = get_all_categories()
main_categories = get_main_categories()
subcategories = get_subcategories_list()

In [4]:
# Accessing podcast and episode data
podcasts = sporc.get_all_podcasts() # List of Podcast objects
episodes = sporc.get_all_episodes() # List of Episode objects


________

### Call-to-Action Detection Pipeline

Call-to-Action (CTA) language is an important feature of persuasive communication. In marketing research, CTAs are defined as explicit prompts that encourage audiences to take a desired action, such as making a purchase or clicking on a link. Studies on digital advertising describe them as message components designed to urge consumers toward a specific behavioural response. In communication and CSR research, CTAs appear as elements that ask audiences to engage with a cause or organization, for example by donating or sharing content. From the perspective of speech-act theory, these correspond to directive speech acts where the speaker aims to influence the listener’s future actions. Because CTAs reveal how podcast hosts engage audiences, promote content or sponsors, and attempt to mobilize behaviour, detecting them is valuable for understanding the communicative and commercial structure of episodes.

> <font color="green"> Sources (not thoroughly checked): </font>  
> - [ScienceDirect](https://www.sciencedirect.com/science/article/abs/pii/S1094996818300707?)  
> - [MDPI](https://www.mdpi.com/2071-1050/13/7/3812)  
> - [ResearchGate](https://doi.org/10.15294/lc.v15i1.26029)

For this project we aim to detect whether a **turn** contains at least one CTA. Although CTAs vary widely in form and purpose, we use a binary distinction for reliability and interpretability:

- **CTA**: at least one utterance encouraging the listener to perform a specific action (for example visiting a website, subscribing, donating, or following a link).  
- **Non-CTA**: no such directive.

Binary classification is appropriate because CTAs are relatively sparse and often ambiguous. Developing fine-grained categories, such as differentiating promotional CTAs from engagement or civic CTAs, would require extensive annotation guidelines and substantial training data. Given limited annotation capacity, fine-grained distinctions would be unreliable, whereas a binary label is robust and fully sufficient for downstream analyses.

CTA identification is challenging: some cases are ambiguous or context dependent, CTAs directed at co-hosts rather than listeners ideally should not be counted, and softly phrased CTAs can be difficult to detect. Whisper transcription errors may obscure key cues, and very long turns can embed CTAs among unrelated material. Despite these limitations, our pipeline aims to provide a practical and interpretable approach to identifying directive behaviour in podcast speech.

### Call-to-Action Detection Pipeline

Call-to-Action (CTA) language is an important feature of persuasive communication. In marketing research, CTAs are defined as explicit prompts that encourage audiences to take a desired action, such as making a purchase or clicking on a link. Studies on digital advertising describe them as message components designed to urge consumers toward a specific behavioural response. In communication and CSR research, CTAs appear as elements that ask audiences to engage with a cause or organization, for example by donating or sharing content. From the perspective of speech-act theory, these correspond to directive speech acts where the speaker aims to influence the listener’s future actions. Because CTAs reveal how podcast hosts engage audiences, promote content or sponsors, and attempt to mobilize behaviour, detecting them is valuable for understanding the communicative and commercial structure of episodes.

> <font color="green">Indicative sources (not thoroughly checked):</font>  
> - [ScienceDirect](https://www.sciencedirect.com/science/article/abs/pii/S1094996818300707?)  
> - [MDPI](https://www.mdpi.com/2071-1050/13/7/3812)  
> - [ResearchGate](https://doi.org/10.15294/lc.v15i1.26029)

Our goal is to determine **whether a given turn contains at least one CTA**. Although CTAs often occur at the level of individual sentences, we assign labels at the **turn level** because this is the unit used throughout our conversational analysis. A turn-level label is also more interpretable for downstream episode- and speaker-level summaries. At the same time, very long turns in our dataset can exceed LLM context limits, and CTA cues are usually expressed within single sentences rather than spread across an entire turn. This motivates our later choice to *detect CTAs at the sentence level and then aggregate the results back to the turn level*.

We therefore use a binary distinction for reliability and interpretability:
- **CTA**: at least one utterance encouraging the listener to perform a specific action (for example visiting a website, subscribing, donating, or following a link).  
- **Non-CTA**: no such directive.

Binary classification is appropriate because CTAs are relatively sparse and often ambiguous. Developing fine-grained categories, such as differentiating promotional CTAs from engagement or civic CTAs, would require extensive annotation guidelines and substantial training data. Given limited annotation capacity, fine-grained distinctions would be unreliable, whereas a binary label is robust and fully sufficient for downstream analyses.

CTA identification is challenging: some cases are ambiguous or context dependent, CTAs directed at co-hosts rather than listeners ideally should not be counted, and softly phrased CTAs can be difficult to detect. Whisper transcription errors may obscure key cues, and very long turns can embed CTAs among unrelated material. Despite these limitations, our pipeline provides a practical and interpretable approach to identifying directive behaviour in podcast speech.


In [13]:
episodes_df = pd.read_parquet("data/episodes.parquet")
turns_df = pd.read_parquet("data/turns.parquet")
episode_lookup = {ep.mp3_url: ep for ep in episodes}
episodes = (episodes_df['mp3_url'].map(episode_lookup)).to_list()

#### Detecting CTA Language

There is no established off-the-shelf method for detecting CTA language in conversational podcast transcripts. [Recent work](https://doi.org/10.48550/arXiv.2409.02690) has explored prompting large language models (LLMs) to classify persuasive or CTA-like statements directly, showing that LLMs can detect subtle directive language. In practice, CTA cues almost always appear within individual sentences, not across entire turns, and some turns in podcast transcripts are extremely long. This means that running an LLM on full turns may detection quality and while often being infeasible due to context window limits. For this reason we will detect CTAs on sentence-level and then aggregate to the turn level. However, running an LLM on every sentence or turn in a large corpus is infeasible due to computational cost and time restrictions. 

To balance semantic quality and scalability, we adopt a hybrid, two-stage pipeline:

1. **Rule-based sentence-level flagging** to identify candidate CTA sentences using cheap lexical and structural cues.  
2. **LLM-based sentence-level classification** applied only to flagged sentences, followed by aggregation back to the turn level.

This design lets us use the LLM where it is most effective while avoiding the prohibitive cost of classifying all sentences.

#### Rule-Based CTA Flagging

In the first stage, we use a rule-based approach grounded in linguistic cues, lexical patterns, and URL-like expressions to flag sentences that might contain CTAs. Since no standard CTA lexicon exists for conversational speech, we developed domain-specific lexicons iteratively by:

- surveying common CTA verbs and phrases used in podcast intros, outros, and sponsor segments  
- reviewing marketing-oriented CTA verb lists (for example subscribe, follow, join, sign up, visit)  
- inspecting a sample of turns containing URLs or sponsor mentions  
- including multi-word expressions such as *"sign up"* and *"use code"* that frequently appear in spoken promotions  
- adding Whisper-style URL distortions such as *"dot com"* and *"slash slash"* to capture transcription artifacts

We then apply NLTK to split turns into sentences and use simple heuristics (CTA lexicon matches, URL-like patterns, and imperative-style structures) to flag sentences that are likely to contain CTA language. This stage is intentionally broad and inclusive: its primary goal is to capture as many potential CTAs as possible, even at the cost of false positives, so that the LLM in the next stage can focus on a much smaller subset of sentences.

The output of this step is a set of sentences marked as *candidate CTA sentences*, along with their turn indices, so that we can later aggregate back to the turn level.

In [ ]:
import re
import nltk
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
from nltk import word_tokenize, pos_tag
from nltk.tokenize import sent_tokenize
from tqdm.auto import tqdm
import torch
from transformers import AutoTokenizer
tqdm.pandas()

In [7]:
CTA_VERBS = {
    "subscribe", "follow", "share", "like", "rate", "review",
    "visit", "check", "sign", "donate", "support", "join",
    "buy", "use", "download", "click", "call", "email",
    "register", "tell", "spread"
}

CTA_MULTIWORD = {
    "sign up", "log in", "check out"
}

CTA_PHRASES = {
    "use code", "promo code", "discount code", "free trial",
    "link in", "show notes","dot com", "dot org", "visit us at"
}

URL_PATTERN = re.compile(
    r"https?://\S+|"
    r"www\.\S+|"
    r"\b\w+\.(com|org|net|io|co)\b|"
    r"http\s+colon|"
    r"slash\s+slash",
    re.IGNORECASE
)

SECOND_PERSON = {"you", "your", "yours"}

def has_second_person(tokens):
    return any(t in SECOND_PERSON for t in tokens)

def has_cta_lexicon(tokens):
    """Return True only if a CTA verb appears in a CTA-like syntactic position."""
    if any(t in CTA_VERBS for t in tokens):
        return True
    
    return False

def is_imperative(tokens, pos_tags):
    """Very simple imperative heuristic:
    Sentence starts with a base form verb (VB) or 'please' followed by VB.
    """
    if not tokens:
        return False
    
    if pos_tags[0][1] == "VB":
        return True
    
    if tokens[0] == "please" and len(pos_tags) > 1 and pos_tags[1][1] == "VB":
        return True

    return False

In [ ]:
def is_cta_sentence(sentence):
    """Return True if sentence contains CTA-like content."""
    if not isinstance(sentence, str) or not sentence.strip():
        return False

    text = sentence.lower()
    if URL_PATTERN.search(text):
        return True
    
    for phrase in CTA_PHRASES:
        if phrase in text:
            return True
        
    for phrase in CTA_MULTIWORD:
        if phrase in text:
            return True

    # Token + POS-level cues
    tokens = word_tokenize(text)
    pos_tags = pos_tag(tokens)
    
    if is_imperative(tokens, pos_tags) and has_cta_lexicon(tokens):
        return True

    if has_cta_lexicon(tokens) and has_second_person(tokens):
        # exclude questions directed at co-hosts
        if not text.strip().endswith("?"):
            return True

    return False



def count_sentences(turn_text):
    """Count number of sentences in the turn."""
    if not isinstance(turn_text, str) or not turn_text.strip():
        return 0
    text = turn_text.strip()
    if not any(p in text for p in ".?!"):
        return 1
    return len(sent_tokenize(text))

def get_flagged_sentences(turn_text, turn_id, turn_index):
    """Return flagged CTA candidate sentences for a given turn as a list of dicts."""
    if not isinstance(turn_text, str) or not turn_text.strip():
        return []

    text = turn_text.strip()
    sentences = sent_tokenize(text) if any(p in text for p in ".?!") else [text]

    flagged = []
    for idx, s in enumerate(sentences):
        if is_cta_sentence(s):
            flagged.append({
                "sentence": s,
                "mp3_url": turn_id, 
                "turn_index": turn_index, 
                "sent_idx": idx 
            })

    return flagged


def classify_turn_cta(turn_text):
    """Return True if any sentence in the turn contains a potential CTA."""
    if not isinstance(turn_text, str) or not turn_text.strip():
        return False

    text = turn_text.strip()

    if not any(p in text for p in ".?!"):
        return is_cta_sentence(text)

    sentences = sent_tokenize(text)
    return any(is_cta_sentence(s) for s in sentences)

In [17]:
turns_df['flagged'] = turns_df['clean_text'].progress_apply(
    lambda x: classify_turn_cta(x) if isinstance(x, str) else False
)
turns_df['flagged_sentences'] = turns_df.progress_apply(
    lambda row: get_flagged_sentences(
        row['clean_text'], 
        row['mp3_url'], 
        row['turn_index']
    ) if isinstance(row['clean_text'], str) else [],
    axis=1
)

turns_df['num_flagged'] = turns_df['flagged_sentences'].apply(len)
turns_df['num_sentences'] = turns_df['clean_text'].progress_apply(
    lambda x: count_sentences(x)
)
turns_df.head()

  0%|          | 0/199491 [00:00<?, ?it/s]

  0%|          | 0/199491 [00:00<?, ?it/s]

  0%|          | 0/199491 [00:00<?, ?it/s]

,episode_title,turn_index,speaker,start_time,end_time,duration,raw_text,clean_text,is_removed,num_words,mp3_url,flagged,flagged_sentences,num_flagged,num_sentences
0,Best of SingOut SpeakOut No.3,0,SPEAKER_00,0.00,60.00,60.00,I'm Simon Shapiro and this is Sing Out Speak ...,I'm Simon Shapiro and this is Sing Out Speak O...,False,124,https://www.buzzsprout.com/783020/4252475-best...,True,[{'mp3_url': 'https://www.buzzsprout.com/78302...,1,8
1,It's All Gone,0,SPEAKER_00,0.00,78.16,78.16,I'm Simon Shapiro and this is Sing Out Speak ...,I'm Simon Shapiro and this is Sing Out Speak O...,False,172,https://www.buzzsprout.com/783020/4165286-it-s...,True,[{'mp3_url': 'https://www.buzzsprout.com/78302...,2,9
2,It's All Gone,1,SPEAKER_02,78.16,115.31,37.15,Music] [Music] [Music] [Music] [Music] [,None,True,0,https://www.buzzsprout.com/783020/4165286-it-s...,False,[],0,0
3,It's All Gone,2,SPEAKER_01,115.31,360.16,244.85,Music] [Music] [Music] [Music] [Music] [Music]...,None,True,0,https://www.buzzsprout.com/783020/4165286-it-s...,False,[],0,0
4,Today Is Yesterday,0,SPEAKER_05,0.00,36.99,36.99,I'm Simon Shapiro and this is Sing Out Speak ...,I'm Simon Shapiro and this is Sing Out Speak O...,False,77,https://www.buzzsprout.com/783020/3983942-toda...,True,[{'mp3_url': 'https://www.buzzsprout.com/78302...,1,3


In [18]:
print(f"Proportion of turns flagged as CTA: {turns_df['flagged'].mean():.1%}")

valid = turns_df[turns_df['num_sentences'] > 0]
avg_cta_sentence_prop = (valid['num_flagged'] / valid['num_sentences']).mean()
print(f"Average flagged sentence proportion per turn: {avg_cta_sentence_prop:.1%}")

Proportion of turns flagged as CTA: 9.4%
Average flagged sentence proportion per turn: 4.6%


In [21]:
turns_df.to_parquet("data/turns_cta.parquet", index=False)
turns_df = pd.read_parquet("data/turns_cta.parquet")

#### LLM-Based CTA Classification

In the second stage, we refine the rule-based predictions using a causal LLM. Although fine-tuning a smaller supervised classifier such as BERT or RoBERTa would likely produce the best performance, we do not have annotated data or the resources for supervised training. We therefore rely on few-shot prompting to leverage the semantic understanding of a causal LLM.

We use a few-shot prompt with representative CTA and non-CTA examples tailored to the model's expected input style to ensure consistent outputs.

The LLM is applied only to sentences flagged by the rule-based detector, which keeps computational cost manageable while preserving high recall. Each flagged sentence receives a binary CTA label, and these sentence-level predictions are then aggregated to the turn level. A turn is considered **CTA** if at least one of its sentences is classified as containing a CTA.

In [44]:
# Create sentence level dataframe with the flagged stentences
turns_df = pd.read_parquet("data/turns_cta.parquet")
sent_df = turns_df['flagged_sentences'].explode().dropna().to_frame()

sent_df["mp3_url"] = sent_df["flagged_sentences"].apply(lambda d: d["mp3_url"])
sent_df["turn_index"] = sent_df["flagged_sentences"].apply(lambda d: d["turn_index"])
sent_df["sentence"] = sent_df["flagged_sentences"].apply(lambda d: d["sentence"])
sent_df["sent_idx"] = sent_df["flagged_sentences"].apply(lambda d: d["sent_idx"])

sent_df = sent_df.drop(columns=['flagged_sentences'])
print(f"Number of flagged sentences: {len(sent_df):,}")
sent_df.head()

Number of flagged sentences: 29,116


,mp3_url,turn_index,sentence,sent_idx
0,https://www.buzzsprout.com/783020/4252475-best...,0,My will to connect with you and share the best...,2
1,https://www.buzzsprout.com/783020/4165286-it-s...,0,My will to connect with you and share the best...,2
1,https://www.buzzsprout.com/783020/4165286-it-s...,0,I hope you like it and I hope you have a fanta...,7
4,https://www.buzzsprout.com/783020/3983942-toda...,0,My will to connect with you and share the best...,2
7,https://www.buzzsprout.com/783020/3983942-toda...,3,But I know it can be tough when you're stuck i...,3


Podcasts tend to have repetive parts: intros, outroes, sponser segments

In [45]:
sent_df["sentence"][0],\
sent_df["sentence"][4]

("My will to connect with you and share the best of me has grown greater than any fear and that's why I am compelled to Sing Out Speak Out.",
 "My will to connect with you and share the best of me has grown greater than any fear and that's why I am compelled to Sing Out Speak Out")

In [46]:
import string

def normalize_for_matching(s):
    if not isinstance(s, str):
        return s
    # Remove punctuation
    s2 = re.sub(f"[{re.escape(string.punctuation)}]", "", s)
    return s2.strip().lower()

sent_df["sentence_key"] = sent_df["sentence"].apply(normalize_for_matching)
sentence_counts = sent_df["sentence_key"].value_counts()

sent_df["is_repeat"] = sent_df["sentence_key"].map(lambda k: sentence_counts[k] > 1)
sent_df["num_occur"] = sent_df["sentence_key"].map(sentence_counts)
sent_df.head()

,mp3_url,turn_index,sentence,sent_idx,sentence_key,is_repeat,num_occur
0,https://www.buzzsprout.com/783020/4252475-best...,0,My will to connect with you and share the best...,2,my will to connect with you and share the best...,True,5
1,https://www.buzzsprout.com/783020/4165286-it-s...,0,My will to connect with you and share the best...,2,my will to connect with you and share the best...,True,5
1,https://www.buzzsprout.com/783020/4165286-it-s...,0,I hope you like it and I hope you have a fanta...,7,i hope you like it and i hope you have a fanta...,False,1
4,https://www.buzzsprout.com/783020/3983942-toda...,0,My will to connect with you and share the best...,2,my will to connect with you and share the best...,True,5
7,https://www.buzzsprout.com/783020/3983942-toda...,3,But I know it can be tough when you're stuck i...,3,but i know it can be tough when youre stuck in...,False,1


In [50]:
unique_sent_df = sent_df.drop_duplicates(subset=["sentence_key"]).copy()
unique_sent_df = unique_sent_df[["sentence", "sentence_key"]] # only keep needed columns
num_repeated_instances = sent_df["is_repeat"].sum()
total_instances = len(sent_df)

print(f"Repeated sentence instances: {num_repeated_instances:,} "
      f"({num_repeated_instances / total_instances:.1%})")

Repeated sentence instances: 1,853 (6.4%)


In [ ]:
unique_sent_df.to_parquet("data/sentences.parquet", index=False)

We run our CTA classifier using the **vLLM** library because it provides substantial performance and scalability benefits compared to the standard HuggingFace generation pipeline. vLLM is designed for high-throughput inference and implements an optimized attention backend (PagedAttention) that allows the model to reuse computation efficiently, especially when many prompts share the same prefix (as is the case in our few-shot classification setup). This reduces both memory usage and latency while preserving exact model behavior.

In testing, this improved inference speed by more than a factor of **15**!

Using vLLM allows us to classify a large number of sentences quickly, reliably, and at a fraction of the cost of naïve generation. This makes it practical to apply an LLM-based classifier across our full dataset without exceeding resource constraints.

We still use the HuggingFace `AutoTokenizer` to construct the prompts via its chat template. This ensures that prompts follow the exact formatting expected by the model while delegating the actual tokenization and generation to vLLM for efficient execution. 

We load the model and tokenizer:

In [ ]:
!pip install vllm -q

In [ ]:
from vllm import LLM, SamplingParams

In [ ]:
MODEL_NAME = "microsoft/Phi-3-mini-128k-instruct" # 4B param model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side='left')
llm = LLM(MODEL_NAME)

#### Prompting

In [108]:
PHI3_SYSTEM_PROMPT = (
    "You are a text classification assistant. "
    "Your task is to determine whether a sentence contains a Call to Action (CTA). "
    "A CTA is any utterance encouraging the listening audience to perform a specific action. "
    "Respond ONLY with TRUE or FALSE."
)

FEW_SHOT_EXAMPLES = [
    {
        "role": "user",
        "content": "Sentence: If you enjoy this podcast, please subscribe and leave a review.\nCTA?"
    },
    {"role": "assistant", "content": "TRUE"},

    {
        "role": "user",
        "content": "Sentence: Climate change is a complex issue with many drivers.\nCTA?"
    },
    {"role": "assistant", "content": "FALSE"},

    {
        "role": "user",
        "content": "Sentence: There's a link in the show notes if you want to support the campaign.\nCTA?"
    },
    {"role": "assistant", "content": "TRUE"},

    {
        "role": "user",
        "content": "Sentence: Thanks for listening, see you next time.\nCTA?"
    },
    {"role": "assistant", "content": "FALSE"},
]

def build_fewshot_prompt(sentence):
    messages = [
        {"role": "system", "content": PHI3_SYSTEM_PROMPT},
        *FEW_SHOT_EXAMPLES,
        {
            "role": "user",
            "content": f"Sentence: {sentence}\nCTA?"
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

def prompt_token_length(prompt, tokenizer):
    return len(tokenizer(prompt, add_special_tokens=False)["input_ids"])

In [109]:
df = unique_sent_df.copy() # Shorten name to make code more readable
df["promt"] = df["sentence"].progress_apply(build_fewshot_prompt)
df["prompt_len"] = df["promt"].progress_apply(lambda p: prompt_token_length(p, tokenizer))

MAX_LEN = 4096  # Phi-3-mini limit
df["too_long"] = df["prompt_len"] > MAX_LEN
print(f"Dataframe has {df["too_long"].sum()} rows that exceed the context window")

  0%|          | 0/27976 [00:00<?, ?it/s]

  0%|          | 0/27976 [00:00<?, ?it/s]

Dataframe has 14 rows that exceed the context window


In [ ]:
params = SamplingParams(
    temperature=0.0,
    top_p=1.0,
    max_tokens=3,
)

def run_vllm(prompts, llm, params):
    outputs = llm.generate(prompts, params)
    return [o.outputs[0].text.strip() for o in outputs]

def extract_bool(text):
    t = text.strip().lower()
    if t.startswith("true"):
        return True
    if t.startswith("false"):
        return False
    return None

In [ ]:
valid_df = df[~df["too_long"]].copy()

prompts = valid_df["promt"].tolist()
llm_outputs = run_vllm(prompts, llm, params) # Run in colab on L4 GPU
valid_df["llm_out"] = llm_outputs
valid_df["has_cta"] = valid_df["llm_out"].apply(extract_bool)

In [ ]:
# Insert predictions back into full df
df.loc[~df["too_long"], "llm_out"] = valid_df["llm_out"]
df.loc[~df["too_long"], "has_cta"] = valid_df["has_cta"]

# Long prompts get default
df.loc[df["too_long"], "llm_out"] = None
df.loc[df["too_long"], "has_cta"] = None 

In [ ]:
#df.to_parquet("data/sentences_llm.parquet", index=False)
df = pd.read_parquet("data/sentences_llm.parquet")
df.head()

,sentence,sentence_key,promt,prompt_len,too_long,llm_out,has_cta
0,My will to connect with you and share the best...,my will to connect with you and share the best...,<|system|>\nYou are a text classification assi...,197,False,FALSE,False
1,I hope you like it and I hope you have a fanta...,i hope you like it and i hope you have a fanta...,<|system|>\nYou are a text classification assi...,178,False,FALSE,False
2,But I know it can be tough when you're stuck i...,but i know it can be tough when youre stuck in...,<|system|>\nYou are a text classification assi...,244,False,FALSE,False
3,"If you like the song, it's available at all th...",if you like the song its available at all the ...,<|system|>\nYou are a text classification assi...,297,False,TRUE,True
4,If you like the,if you like the,<|system|>\nYou are a text classification assi...,167,False,FALSE,False


In [113]:
# Aggregate back to dublicate sentences
sent_df = sent_df.merge(
    df[["sentence_key", "has_cta"]],
    on="sentence_key",
    how="left"
)

cta_sentence_lists = (
    sent_df[sent_df["has_cta"] == True]
    .groupby(["mp3_url", "turn_index"])["sentence"]
    .apply(list)
    .reset_index()
    .rename(columns={"sentence": "cta_sentences"})
)

In [119]:
sent_df

,mp3_url,turn_index,sentence,sent_idx,sentence_key,is_repeat,num_occur,has_cta
0,https://www.buzzsprout.com/783020/4252475-best...,0,My will to connect with you and share the best...,2,my will to connect with you and share the best...,True,5,False
1,https://www.buzzsprout.com/783020/4165286-it-s...,0,My will to connect with you and share the best...,2,my will to connect with you and share the best...,True,5,False
2,https://www.buzzsprout.com/783020/4165286-it-s...,0,I hope you like it and I hope you have a fanta...,7,i hope you like it and i hope you have a fanta...,False,1,False
3,https://www.buzzsprout.com/783020/3983942-toda...,0,My will to connect with you and share the best...,2,my will to connect with you and share the best...,True,5,False
4,https://www.buzzsprout.com/783020/3983942-toda...,3,But I know it can be tough when you're stuck i...,3,but i know it can be tough when youre stuck in...,False,1,False
...,...,...,...,...,...,...,...,...
29111,https://anchor.fm/s/a9cd2a4/podcast/play/14615...,61,"And sure enough, there's this wonderful lady w...",4,and sure enough theres this wonderful lady who...,False,1,False
29112,https://anchor.fm/s/a9cd2a4/podcast/play/14615...,78,"Well, if we were going to emphasize water bapt...",2,well if we were going to emphasize water bapti...,False,1,False
29113,https://anchor.fm/s/a9cd2a4/podcast/play/14615...,80,"I'll tell you that, to cut through the",13,ill tell you that to cut through the,False,1,False
29114,https://anchor.fm/s/a9cd2a4/podcast/play/14615...,82,"Yeah, and I've heard stories in the Scandinavi...",1,yeah and ive heard stories in the scandinavian...,False,1,False


In [115]:
# Aggregating back to turns_df
turn_llm_cta = sent_df.groupby(['mp3_url', 'turn_index'])['has_cta'].any()
turn_llm_cta_count = sent_df.groupby(['mp3_url', 'turn_index'])['has_cta'].sum()

turns_df = turns_df.merge(
    turn_llm_cta.rename("has_cta"),
    on=["mp3_url", "turn_index"],
    how="left"
)

turns_df = turns_df.merge(
    turn_llm_cta_count.rename("num_cta_sentences"),
    on=["mp3_url", "turn_index"],
    how="left"
)

turns_df = turns_df.merge(
    cta_sentence_lists,
    on=["mp3_url", "turn_index"],
    how="left"
)

In [120]:
turns_df.head()

,episode_title,turn_index,speaker,start_time,end_time,duration,raw_text,clean_text,is_removed,num_words,mp3_url,flagged,flagged_sentences,num_flagged,num_sentences,has_cta,num_cta_sentences,cta_sentences
0,Best of SingOut SpeakOut No.3,0,SPEAKER_00,0.00,60.00,60.00,I'm Simon Shapiro and this is Sing Out Speak ...,I'm Simon Shapiro and this is Sing Out Speak O...,False,124,https://www.buzzsprout.com/783020/4252475-best...,True,[{'mp3_url': 'https://www.buzzsprout.com/78302...,1,8,False,False,NaN
1,It's All Gone,0,SPEAKER_00,0.00,78.16,78.16,I'm Simon Shapiro and this is Sing Out Speak ...,I'm Simon Shapiro and this is Sing Out Speak O...,False,172,https://www.buzzsprout.com/783020/4165286-it-s...,True,[{'mp3_url': 'https://www.buzzsprout.com/78302...,2,9,False,0,NaN
2,It's All Gone,1,SPEAKER_02,78.16,115.31,37.15,Music] [Music] [Music] [Music] [Music] [,None,True,0,https://www.buzzsprout.com/783020/4165286-it-s...,False,[],0,0,NaN,NaN,NaN
3,It's All Gone,2,SPEAKER_01,115.31,360.16,244.85,Music] [Music] [Music] [Music] [Music] [Music]...,None,True,0,https://www.buzzsprout.com/783020/4165286-it-s...,False,[],0,0,NaN,NaN,NaN
4,Today Is Yesterday,0,SPEAKER_05,0.00,36.99,36.99,I'm Simon Shapiro and this is Sing Out Speak ...,I'm Simon Shapiro and this is Sing Out Speak O...,False,77,https://www.buzzsprout.com/783020/3983942-toda...,True,[{'mp3_url': 'https://www.buzzsprout.com/78302...,1,3,False,False,NaN


In [118]:
turns_df[turns_df["has_cta"]==True].head()

,episode_title,turn_index,speaker,start_time,end_time,duration,raw_text,clean_text,is_removed,num_words,mp3_url,flagged,flagged_sentences,num_flagged,num_sentences,has_cta,num_cta_sentences,cta_sentences
15,Today Is Yesterday,11,SPEAKER_05,376.28,408.28,32.00,"If you like the song, it's available at all t...","If you like the song, it's available at all th...",False,117,https://www.buzzsprout.com/783020/3983942-toda...,True,[{'mp3_url': 'https://www.buzzsprout.com/78302...,1,1,True,True,"[If you like the song, it's available at all t..."
27,Saturn Return,10,SPEAKER_03,436.12,468.49,32.37,song it's available at all the places you usu...,song it's available at all the places you usua...,False,113,https://www.buzzsprout.com/783020/3892169-satu...,True,[{'mp3_url': 'https://www.buzzsprout.com/78302...,3,5,True,1,[If you've got something out of the show then ...
61,Episode 33 'California’s Higher Education Lan...,17,SPEAKER_01,489.87,557.32,67.45,earlier our work is really around systems cha...,earlier our work is really around systems chan...,False,177,http://dts.podtrac.com/redirect.mp3/feeds.soun...,True,[{'mp3_url': 'http://dts.podtrac.com/redirect....,1,2,True,True,[earlier our work is really around systems cha...
72,Episode 33 'California’s Higher Education Lan...,28,SPEAKER_00,695.07,768.08,73.01,right so on that theme I mean obviously you'v...,right so on that theme I mean obviously you've...,False,183,http://dts.podtrac.com/redirect.mp3/feeds.soun...,True,[{'mp3_url': 'http://dts.podtrac.com/redirect....,1,1,True,True,[right so on that theme I mean obviously you'v...
96,Episode 33 'California’s Higher Education Lan...,52,SPEAKER_00,1044.43,1105.68,61.25,right no I mean I certainly hope that we take...,right no I mean I certainly hope that we take ...,False,161,http://dts.podtrac.com/redirect.mp3/feeds.soun...,True,[{'mp3_url': 'http://dts.podtrac.com/redirect....,1,1,True,True,[right no I mean I certainly hope that we take...


In [ ]:
turns_df.to_parquet("data/turns_cta.parquet", index=False)

_____